In [ ]:
# ===== 셀 1: 연습용 가짜 메일을 읽어서 칸별로 출력하기 =====
from email import message_from_string
import json
# 하는 일: 메일을 읽는 도구(message_from_string)와 결과를 출력할 도구(json)를 불러옴
# 이유: 둘 다 파이썬에 기본으로 들어 있어서 설치(pip install) 없이 바로 쓸 수 있음

# 연습용 가짜 메일 원문 (실제 메일 아님, .example은 문서용 가짜 주소)
raw_email = """From: "HR Team" <hr@company.example>
Reply-To: <hr-check@company-notice.example>
To: <staff@company.example>
Subject: [Notice] Please verify your employee account
Authentication-Results: mx.company.example; spf=fail smtp.mailfrom=company.example; dkim=none; dmarc=fail

Please log in and verify your account by today.
"""
# 하는 일: 메일 원문을 글자 덩어리로 저장함 (따옴표 세 개 """ 사이는 여러 줄 글자)
# 메일 구조: 맨 위 "칸이름: 값" 줄들이 헤더(봉투에 적힌 정보), 빈 줄 아래가 본문(편지 내용)
# 이 가짜 메일의 함정 2개:
#   ① From은 company.example인데 Reply-To는 company-notice.example (답장이 다른 곳으로 감)
#   ② 받는 서버가 검사해 보니 spf=fail (허락받지 않은 서버에서 보냄)

msg = message_from_string(raw_email)
print("보낸 사람(From)   :", msg["From"])
print("회신 주소(Reply-To):", msg["Reply-To"])
print("검사 결과         :", msg["Authentication-Results"])
# 하는 일: 원문을 읽어서 msg["From"]처럼 칸 이름으로 꺼낼 수 있게 만들고 세 칸을 출력함
# 이유: 피싱 메일은 본문보다 헤더에서 거짓말이 드러나는 경우가 많아서 헤더 읽는 법부터 연습

보낸 사람(From)   : "HR Team" <hr@company.example>
회신 주소(Reply-To): <hr-check@company-notice.example>
검사 결과         : mx.company.example; spf=fail smtp.mailfrom=company.example; dkim=none; dmarc=fail


In [ ]:
# ===== 셀 2: 이메일 분석 함수 =====
from email.utils import parseaddr

def get_domain(header_value):
    name, address = parseaddr(header_value or "")
    if "@" not in address:
        return None
    return address.split("@")[-1].lower()
# 하는 일: '"HR Team" <hr@company.example>' 같은 칸에서 메일 주소만 꺼내 @ 뒷부분(도메인)만 돌려줌
#          예: "HR Team" <hr@company.example> → company.example
# 이유: 보낸 사람과 회신 주소의 "회사 이름 부분"이 같은지 비교하려고
# 참고: 칸이 비었거나 주소 모양이 아니면 None(없음)을 돌려줌

def analyze_email_sample(raw_text, filename):
    reasons = []        # 점수가 나온 이유를 모아 두는 목록 (URL 함수와 똑같음)
    unavailable = []    # 확인하지 못한 항목을 모아 두는 목록 (URL 함수와 똑같음)
    msg = message_from_string(raw_text)

    # 1. SPF 실패 (임시 40점, B가 확정)
    auth = msg.get("Authentication-Results")
    if auth is None:
        unavailable.append("spf")
    elif "spf=fail" in auth.lower():
        reasons.append({
            "factor": "SPF 실패",
            "points": 40,
            "text": "보낸 사람 도메인이 허용하지 않은 서버에서 발송된 메일입니다."
        })
    # 하는 일: 받는 서버가 적어 둔 검사 결과 칸에서 spf=fail을 찾으면 점수를 줌
    # 이유: SPF는 "이 도메인으로는 이 서버들만 메일을 보낼 수 있다"는 등록 목록이라,
    #       fail이면 허락받지 않은 서버가 회사 이름을 빌려 보냈다는 뜻
    # 참고: 검사 결과 칸이 아예 없으면 0점이 아니라 unavailable에 적음 (URL의 whois와 같은 약속)
    # 참고: 지금은 결과를 "읽기만" 함. 직접 DNS를 조회해 검사하는 건 다음 단계에서 dnspython으로 구현

    # 2. 보낸 사람 불일치 (임시 30점, B가 확정)
    from_domain = get_domain(msg.get("From"))
    reply_domain = get_domain(msg.get("Reply-To"))
    if from_domain and reply_domain and from_domain != reply_domain:
        reasons.append({
            "factor": "보낸 사람 불일치",
            "points": 30,
            "text": "화면에 보이는 보낸 사람(From)과 실제 회신 주소(Reply-To)의 도메인이 다릅니다."
        })
    # 하는 일: 보낸 사람 도메인과 회신 주소 도메인이 둘 다 있고 서로 다르면 점수를 줌
    # 이유: "인사팀"인 척 보내고 답장은 공격자 주소로 받아 가는 수법이 많기 때문
    # 참고: Reply-To가 없는 메일은 정상적으로 흔해서 "확인 못 함"이 아니라 그냥 넘어감

    # 3. 점수 합치기
    score = sum(r["points"] for r in reasons)
    # 이유: README 약속 1번 "score는 reasons의 points를 더한 값"

    # 4. 팀 약속 모양으로 돌려주기
    return {
        "type": "email",
        "input": filename,
        "score": score,
        "model_probability": None,   # 이메일은 아직 모델이 없어서 비어 있음 (JSON에서는 null)
        "reasons": reasons,
        "unavailable": unavailable
    }
    # 이유: URL 결과와 칸 이름이 똑같아야 B(화면)와 C(장부)가 구분 없이 받아 쓸 수 있음

print("셀 2 준비 완료")

셀 2 준비 완료


In [ ]:
# ===== 셀 3: 샘플 3 결과 만들기 + 약속 확인 =====
sample3 = analyze_email_sample(raw_email, "sample_hr_notice.eml")
print(json.dumps(sample3, ensure_ascii=False, indent=2))
# 하는 일: 가짜 메일을 분석 함수에 넣고 결과를 보기 좋게 출력함
# input 칸: 메일은 링크처럼 주소가 없어서 "파일 이름"을 적음 (README 칸 설명)
#           .eml은 메일 한 통을 파일로 저장할 때 쓰는 확장자

assert sample3["type"] == "email"
assert sample3["score"] == sum(r["points"] for r in sample3["reasons"])
assert sample3["model_probability"] is None
print("약속 확인 통과!")
# 하는 일: assert는 "이게 맞아야 한다"는 검사. 틀리면 빨간 AssertionError가 나고 멈춤
# 확인 1: 이메일 검사 결과라고 표시됐는지
# 확인 2: score가 points 합과 같은지 (README 약속 1번)
# 확인 3: 모델이 없으니 model_probability가 비어 있는지 (README 샘플 설명)

normal_email = """From: "HR Team" <hr@company.example>
To: <staff@company.example>
Subject: [Notice] Office hours
Authentication-Results: mx.company.example; spf=pass smtp.mailfrom=company.example

See you tomorrow.
"""
print("정상 메일 → 점수", analyze_email_sample(normal_email, "normal.eml")["score"])
# 하는 일: SPF 통과(spf=pass), 회신 주소 없음인 정상 메일도 넣어 봄
# 이유: 함수가 아무 메일이나 위험하다고 하지 않는지 확인하려고 (샘플 2에서 진짜 정부 사이트를 넣어 본 것과 같음)

{
  "type": "email",
  "input": "sample_hr_notice.eml",
  "score": 70,
  "model_probability": null,
  "reasons": [
    {
      "factor": "SPF 실패",
      "points": 40,
      "text": "보낸 사람 도메인이 허용하지 않은 서버에서 발송된 메일입니다."
    },
    {
      "factor": "보낸 사람 불일치",
      "points": 30,
      "text": "화면에 보이는 보낸 사람(From)과 실제 회신 주소(Reply-To)의 도메인이 다릅니다."
    }
  ],
  "unavailable": []
}
약속 확인 통과!
정상 메일 → 점수 0


In [ ]:
# ===== 셀 4: 샘플 3을 파일로 저장하고 내 컴퓨터로 내려받기 =====
from google.colab import files

with open("email_spoof.json", "w", encoding="utf-8") as f:
    json.dump(sample3, f, ensure_ascii=False, indent=2)
files.download("email_spoof.json")
# 하는 일: 결과를 email_spoof.json 파일로 저장한 뒤 다운로드 폴더로 내려받음 (샘플 2의 셀 10과 같은 방법)
# ensure_ascii=False: 한글이 이상한 글자로 바뀌지 않게 함
# indent=2: 사람이 읽기 좋게 줄 바꾸고 들여쓰기 함
# 주의: 다운로드 창이 안 뜨면 주소창 오른쪽 "팝업 차단" 아이콘에서 허용

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>